# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/minamalak123/FlyRankIntern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

## 1. My rule and its reason codes

**Rule:** Prioritize content that is old enough to need a refresh and still has meaningful search visibility. The score rewards both staleness and search volume, so high-priority items are pages where a refresh has a measurable opportunity.

**Reason codes:**
- `stale_visible` — the page is at least 180 days since its last update and has at least 300 search impressions in the 90-day window.
- `not_stale` — the page is not yet 180 days since its last update.
- `low_visibility` — the page has fewer than 300 search impressions in the 90-day window.

**Signal checks:** I check staleness and search volume before using them in the rule. The verdict is based on the observed relationship between each signal bucket and the declining outcome in the same snapshot; the outcome is used only for auditing/evaluation, not as a scoring feature.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the repository dataset.
# The repository is public, so this works directly in Colab.
DATA_URL = "https://raw.githubusercontent.com/minamalak123/FlyRankIntern/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

# Keep only the fields needed for this baseline.
required = [
    "content_id",
    "client_id",
    "days_since_last_update",
    "impressions_90d",
    "trend_direction",
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Numeric cleanup.
df["days_since_last_update"] = pd.to_numeric(
    df["days_since_last_update"], errors="coerce"
).fillna(0)

df["impressions_90d"] = pd.to_numeric(
    df["impressions_90d"], errors="coerce"
).fillna(0)

# Audit-only target.
# IMPORTANT: this is never used as a feature in the baseline score.
audit_declining = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)

# -------------------------
# SIGNAL 1: STALENESS
# -------------------------
stale_bins = [-np.inf, 90, 180, 365, np.inf]
stale_labels = ["0-90", "91-180", "181-365", "365+"]

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=stale_bins,
    labels=stale_labels,
    right=True,
)

stale_check = (
    pd.DataFrame({
        "bucket": df["staleness_bucket"],
        "declining": audit_declining,
    })
    .groupby("bucket", observed=False)
    .agg(
        n=("declining", "size"),
        declining_rate=("declining", "mean"),
    )
    .reset_index()
)

print("SIGNAL 1 — STALENESS")
print(stale_check.to_string(index=False))
print(f"n = {len(df):,}")

stale_rates = stale_check["declining_rate"].dropna().tolist()

if len(stale_rates) >= 2:
    low = stale_rates[0]
    high = stale_rates[-1]
    if high - low >= 0.05:
        stale_verdict = "CONFIRMED"
    elif low - high >= 0.05:
        stale_verdict = "OPPOSITE"
    elif max(stale_rates) - min(stale_rates) >= 0.03:
        stale_verdict = "MIXED"
    else:
        stale_verdict = "FALSE"
else:
    stale_verdict = "FALSE"

print(f"VERDICT: {stale_verdict}")

# -------------------------
# SIGNAL 2: SEARCH VOLUME
# -------------------------
volume_bins = [-np.inf, 99, 299, 2999, np.inf]
volume_labels = ["<100", "100-299", "300-2999", "3000+"]

df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=volume_bins,
    labels=volume_labels,
    right=True,
)

volume_check = (
    pd.DataFrame({
        "bucket": df["volume_bucket"],
        "declining": audit_declining,
    })
    .groupby("bucket", observed=False)
    .agg(
        n=("declining", "size"),
        declining_rate=("declining", "mean"),
    )
    .reset_index()
)

print("\nSIGNAL 2 — SEARCH VOLUME")
print(volume_check.to_string(index=False))
print(f"n = {len(df):,}")

volume_rates = volume_check["declining_rate"].dropna().tolist()

if len(volume_rates) >= 2:
    low = volume_rates[0]
    high = volume_rates[-1]
    if high - low >= 0.05:
        volume_verdict = "CONFIRMED"
    elif low - high >= 0.05:
        volume_verdict = "OPPOSITE"
    elif max(volume_rates) - min(volume_rates) >= 0.03:
        volume_verdict = "MIXED"
    else:
        volume_verdict = "FALSE"
else:
    volume_verdict = "FALSE"

print(f"VERDICT: {volume_verdict}")

print("\nAudit note:")
print(
    "The declining outcome is used only to test whether the signals show a "
    "directional relationship. It is NOT included in the ranking score."
)

SIGNAL 1 — STALENESS
 bucket     n  declining_rate
   0-90 20655        0.512031
 91-180  9171        0.611057
181-365   169        0.467456
   365+     5        0.600000
n = 30,000
VERDICT: CONFIRMED

SIGNAL 2 — SEARCH VOLUME
  bucket     n  declining_rate
    <100  7994        0.389042
 100-299  3254        0.613399
300-2999 10469        0.614672
   3000+  8283        0.569963
n = 30,000
VERDICT: CONFIRMED

Audit note:
The declining outcome is used only to test whether the signals show a directional relationship. It is NOT included in the ranking score.


## 2. Build the ranked queue

The baseline score is intentionally transparent and has no fitted weights:

- stale = 1 when `days_since_last_update >= 180`
- visible = 1 when `impressions_90d >= 300`
- score = stale × visible × log1p(impressions_90d)

This makes the rule easy to inspect. The action is `refresh` for pages meeting both conditions; otherwise the item is still ranked but receives a lower-priority action label.

No trend, future-window, or target-derived field is used in the score.

In [2]:
# -------------------------
# ONE transparent baseline rule
# -------------------------

df["stale_flag"] = (
    df["days_since_last_update"] >= 180
).astype(int)

df["visible_flag"] = (
    df["impressions_90d"] >= 300
).astype(int)

# Transparent score:
# staleness AND meaningful visibility, weighted only by observed volume.
df["score"] = (
    df["stale_flag"]
    * df["visible_flag"]
    * np.log1p(df["impressions_90d"])
)

# One reason code per row.
df["reason_code"] = np.select(
    [
        (df["stale_flag"] == 1) & (df["visible_flag"] == 1),
        df["stale_flag"] == 0,
    ],
    [
        "stale_visible",
        "not_stale",
    ],
    default="low_visibility",
)

# One action label per row.
df["action"] = np.where(
    df["reason_code"] == "stale_visible",
    "refresh",
    "monitor",
)

# Rank everything.
queue = (
    df[
        [
            "content_id",
            "score",
            "action",
            "reason_code",
            "days_since_last_update",
            "impressions_90d",
        ]
    ]
    .sort_values(
        ["score", "impressions_90d", "days_since_last_update"],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

queue["rank"] = np.arange(1, len(queue) + 1)

# Put rank first.
queue = queue[
    [
        "rank",
        "content_id",
        "score",
        "action",
        "reason_code",
        "days_since_last_update",
        "impressions_90d",
    ]
]

# Required output path.
output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

queue.to_csv(output_path, index=False)

print(f"Rows ranked: {len(queue):,}")
print(f"Refresh actions: {(queue['action'] == 'refresh').sum():,}")
print(f"Monitor actions: {(queue['action'] == 'monitor').sum():,}")
print(f"CSV written to: {output_path}")
print("\nTop 10:")
print(queue.head(10).to_string(index=False))

Rows ranked: 30,000
Refresh actions: 22
Monitor actions: 29,978
CSV written to: work\outputs\baseline_action_score.csv

Top 10:
 rank           content_id     score  action   reason_code  days_since_last_update  impressions_90d
    1 content_cf56e2e2e282 11.029699 refresh stale_visible                     194            61678
    2 content_7368877ea310 10.993278 refresh stale_visible                     194            59472
    3 content_1bfaa38ff26c 10.154869 refresh stale_visible                     194            25715
    4 content_0a91db491d14  9.495519 refresh stale_visible                     193            13299
    5 content_5feee3994adb  8.963544 refresh stale_visible                     194             7812
    6 content_c2d929d83eaa  8.930494 refresh stale_visible                     193             7558
    7 content_b16bd7307b39  8.431853 refresh stale_visible                     194             4590
    8 content_fe16a55cd13d  8.424420 refresh stale_visible              

## 3. Top-20 review

For every top-20 item, I record the action, reason code, a confidence note, and what could make the recommendation wrong. These are decision-support notes rather than claims that the page definitely needs a refresh.

In [3]:
# -------------------------
# TOP-20 REVIEW
# -------------------------

top20 = queue.head(20).copy()

def confidence_note(row):
    if row["reason_code"] == "stale_visible":
        if row["impressions_90d"] >= 3000:
            return "Higher confidence: old content with strong observed search visibility."
        return "Moderate confidence: old content with measurable search visibility."
    return "Lower confidence: the rule does not see both baseline conditions."

def wrong_if(row):
    if row["reason_code"] == "stale_visible":
        return (
            "Wrong if the content is intentionally evergreen, already accurate, "
            "or its search demand is not relevant to a refresh."
        )
    if row["reason_code"] == "not_stale":
        return (
            "Wrong if the page has an urgent quality problem despite being recently updated."
        )
    return (
        "Wrong if the page has valuable demand that is understated by the observed "
        "90-day impression volume."
    )

review = []

for _, row in top20.iterrows():
    review.append({
        "rank": int(row["rank"]),
        "action": row["action"],
        "reason_code": row["reason_code"],
        "confidence_note": confidence_note(row),
        "what_would_make_it_wrong": wrong_if(row),
    })

review_df = pd.DataFrame(review)

print("TOP-20 REVIEW")
print("=" * 120)

for _, row in review_df.iterrows():
    print(
        f"{int(row['rank'])}. "
        f"Action: {row['action']} | "
        f"Reason: {row['reason_code']} | "
        f"Confidence: {row['confidence_note']} | "
        f"Wrong if: {row['what_would_make_it_wrong']}"
    )

TOP-20 REVIEW
1. Action: refresh | Reason: stale_visible | Confidence: Higher confidence: old content with strong observed search visibility. | Wrong if: Wrong if the content is intentionally evergreen, already accurate, or its search demand is not relevant to a refresh.
2. Action: refresh | Reason: stale_visible | Confidence: Higher confidence: old content with strong observed search visibility. | Wrong if: Wrong if the content is intentionally evergreen, already accurate, or its search demand is not relevant to a refresh.
3. Action: refresh | Reason: stale_visible | Confidence: Higher confidence: old content with strong observed search visibility. | Wrong if: Wrong if the content is intentionally evergreen, already accurate, or its search demand is not relevant to a refresh.
4. Action: refresh | Reason: stale_visible | Confidence: Higher confidence: old content with strong observed search visibility. | Wrong if: Wrong if the content is intentionally evergreen, already accurate, or it

## 4. Weak picks + leakage check

The weakest-looking picks are reviewed for cases where the simple rule may over-prioritize a page. A high score does not prove that a refresh will help; it only identifies pages matching the baseline's transparent conditions.

The scoring rule uses only staleness and observed 90-day impressions. It does not use `trend_direction`, `trend_pct`, or any future-window outcome information.

In [4]:
# -------------------------
# WEAK PICKS
# -------------------------

print("WEAK PICK REVIEW")
print("=" * 100)

# Look at the bottom of the positive-scoring group.
positive = queue[queue["score"] > 0].copy()

if len(positive) > 0:
    weak_picks = positive.tail(min(5, len(positive)))

    for _, row in weak_picks.iterrows():
        print(
            f"Rank {int(row['rank'])}: "
            f"{row['content_id']} | "
            f"score={row['score']:.3f} | "
            f"action={row['action']} | "
            f"reason={row['reason_code']} | "
            "Weakness: the rule is simple and may miss page-specific context."
        )
else:
    print("No positive-scoring picks were produced.")

# -------------------------
# LEAKAGE CHECK
# -------------------------

score_features = [
    "days_since_last_update",
    "impressions_90d",
]

forbidden_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
]

print("\nLEAKAGE CHECK")
print("=" * 100)

print("Scoring features:", score_features)

for feature in forbidden_features:
    assert feature not in score_features, f"Leakage detected: {feature}"

print("PASS: no trend/label-derived field is used in the score.")
print("PASS: no future-window field is used in the score.")
print("PASS: content_id is used only as an identifier, not as a feature.")

# Confirm the CSV exists and has the expected core fields.
assert output_path.exists()

required_output_columns = {
    "rank",
    "content_id",
    "score",
    "action",
    "reason_code",
}

assert required_output_columns.issubset(set(queue.columns))

print("PASS: baseline_action_score.csv exists with rank, score, action, and reason_code.")

WEAK PICK REVIEW
Rank 18: content_fd16e3475c29 | score=6.064 | action=refresh | reason=stale_visible | Weakness: the rule is simple and may miss page-specific context.
Rank 19: content_b65fe2792b44 | score=5.919 | action=refresh | reason=stale_visible | Weakness: the rule is simple and may miss page-specific context.
Rank 20: content_ba00ffc6318c | score=5.846 | action=refresh | reason=stale_visible | Weakness: the rule is simple and may miss page-specific context.
Rank 21: content_4729b57ca036 | score=5.817 | action=refresh | reason=stale_visible | Weakness: the rule is simple and may miss page-specific context.
Rank 22: content_6476d1d8c050 | score=5.720 | action=refresh | reason=stale_visible | Weakness: the rule is simple and may miss page-specific context.

LEAKAGE CHECK
Scoring features: ['days_since_last_update', 'impressions_90d']
PASS: no trend/label-derived field is used in the score.
PASS: no future-window field is used in the score.
PASS: content_id is used only as an ident

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.